# Met Museum

### Quick set up and example

Set up of a session with a cache and a cached GET request. As well, there are a few known issues that are important for later:

### Known issues:
1) 403 response after 30 requests: https://github.com/metmuseum/openaccess/issues/60

2) Query parameter order not deterministic: https://github.com/metmuseum/openaccess/issues/51

3) hasImage=true does not always return an object with an image: https://github.com/metmuseum/openaccess/issues/52
It might be worth checking if the opposite is true, so if hasImage=false hides some objects with images.

Restarts the connection after ```ROTATE_EVERY=25``` requests.

In [2]:
import os, json, hashlib
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import itertools

# MET Endpoint
BASE_URL = "https://collectionapi.metmuseum.org/public/collection/v1"

# Create session with polite headers. Might need to close session for issue #60. Known issue 1)
def new_session():
    s = requests.Session()
    # Set up retries because sometimes it fails
    retry = Retry(
        total=5,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))

    # Closing the connection is very important
    s.headers.update({
            "User-Agent": "met-sampling/0.1 (ex@mple.com)",
            "Accept": "application/json",
            "Connection": "close"
        })
    return s

session = new_session()

# Cache set up
CACHE_DIR = "cache/met"
os.makedirs(CACHE_DIR, exist_ok=True)

def _cache_path(name):
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in name)
    return os.path.join(CACHE_DIR, f"{safe}.json")

_request_counter = itertools.count()
ROTATE_EVERY = 25

# Cached fetch function
def fetch_json(url, params=None, cache_name=None, force=False):
    global session

    cache_file = _cache_path(cache_name or hashlib.md5((url + str(params)).encode()).hexdigest())
    if not force and os.path.exists(cache_file):
        with open(cache_file, "r", encoding="utf-8") as f:
            return json.load(f)

    # Close and open a new session after ROTATE_EVERY requests
    i = next(_request_counter)
    if i > 0 and i % ROTATE_EVERY == 0:
        try:
            session.close()
        except Exception:
            pass
        session = new_session()

    r = session.get(url, params=params, timeout=30)
    print(f"GET {r.url} -> {r.status_code}")
    r.raise_for_status()
    data = r.json()

    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    return data

### Example of fetching an object

In [3]:
import pandas as pd

# Object search by ID
def met_object(object_id):
    url = f"{BASE_URL}/objects/{int(object_id)}"
    return fetch_json(url, cache_name=f"object_{object_id}")

# Organize object information as a row. Description is missing, but its in the normal frontend.
def normalize_object(o):
    constituents = o.get("constituents") or []
    first_artist = constituents[0] if constituents else {}
    tags = o.get("tags") or []
    tag_terms = [t.get("term") for t in tags if isinstance(t, dict) and t.get("term")]

    row = {
        "objectID": o.get("objectID"),
        "title": o.get("title"),
        "objectName": o.get("objectName"),
        "classification": o.get("classification"),
        "department": o.get("department"),
        "isHighlight": o.get("isHighlight"),
        "isPublicDomain": o.get("isPublicDomain"),
        "objectURL": o.get("objectURL"),
        "primaryImage": o.get("primaryImage"),
        "primaryImageSmall": o.get("primaryImageSmall"),
        "additionalImages_count": len(o.get("additionalImages") or []),

        # dates
        "objectDate": o.get("objectDate"),
        "objectBeginDate": o.get("objectBeginDate"),
        "objectEndDate": o.get("objectEndDate"),
        "metadataDate": o.get("metadataDate"),

        # medium / size
        "medium": o.get("medium"),
        "dimensions": o.get("dimensions"),

        # geography / culture / period
        "culture": o.get("culture"),
        "country": o.get("country"),
        "region": o.get("region"),
        "city": o.get("city"),
        "period": o.get("period"),
        "dynasty": o.get("dynasty"),
        "reign": o.get("reign"),

        # artist (first constituent)
        "artistDisplayName": o.get("artistDisplayName") or first_artist.get("name"),
        "artistRole": o.get("artistRole") or first_artist.get("role"),
        "artistDisplayBio": o.get("artistDisplayBio"),
        "artistNationality": o.get("artistNationality"),
        "artistBeginDate": o.get("artistBeginDate"),
        "artistEndDate": o.get("artistEndDate"),
        "artistGender": o.get("artistGender") or first_artist.get("gender"),
        "artistWikidata_URL": o.get("artistWikidata_URL") or first_artist.get("constituentWikidata_URL"),
        "artistULAN_URL": o.get("artistULAN_URL") or first_artist.get("constituentULAN_URL"),

        # admin / rights
        "accessionNumber": o.get("accessionNumber"),
        "accessionYear": o.get("accessionYear"),
        "creditLine": o.get("creditLine"),
        "rightsAndReproduction": o.get("rightsAndReproduction"),
        "repository": o.get("repository"),

        # tags
        "tags": "|".join(tag_terms) if tag_terms else None,
        "objectWikidata_URL": o.get("objectWikidata_URL"),
        "GalleryNumber": o.get("GalleryNumber"),
    }
    return row

# Example on "Bashi-Bazouk"
obj = met_object(440723) # https://www.metmuseum.org/art/collection/search/440723
row = normalize_object(obj)
df = pd.DataFrame([row])
df.head(1)

out_csv = f"{CACHE_DIR}/met_art_sample_1.csv"
df.to_csv(out_csv, index=False)

### Query search

In [45]:
# Plain query search in MET archives
# q at the end to mitigate issue #51. Known issue 2)
def met_search(q, **filters):
    url = f"{BASE_URL}/search"

    # build filters first, convert bools to lowercase strings the API expects)
    params = {}
    for k, v in filters.items():
        if isinstance(v, bool):
            params[k] = "true" if v else "false"
        else:
            params[k] = v

    params["q"] = q

    # cache key remains order-insensitive due to sort_keys=True
    cache_key = f"search_{q}_{hashlib.md5(json.dumps(params, sort_keys=True).encode()).hexdigest()}"
    return fetch_json(url, params=params, cache_name=cache_key)

# Search settings
search_query = "painting"
search_filters = dict(
    medium="Paintings",
    hasImages=True,
    isPublicDomain=True,
)

SAMPLE_SIZE = 10

res = met_search(search_query, **search_filters)
ids = res.get("objectIDs") or []
print(f"Found {res.get('total', 0)} candidate IDs.")

rows, kept_ids = [], []
for oid in ids:
    o = met_object(oid)
    if not o.get("isPublicDomain"):
        continue
    if (o.get("classification") or "").lower() != "paintings":
        continue
    if not (o.get("primaryImage") or o.get("primaryImageSmall")):
        continue
    rows.append(normalize_object(o))
    kept_ids.append(oid)
    if len(rows) == SAMPLE_SIZE:
        break

df = pd.DataFrame(rows)
print(f"Collected {len(df)} rows. ObjectIDs: {kept_ids}")

filled = df.notna().sum()
missing = df.isna().sum()
summary = pd.DataFrame({"filled": filled, "missing": missing}).sort_values("missing", ascending=False)

display(summary.T)
out_csv = f"{CACHE_DIR}/met_paintings_sample_10.csv"
df.to_csv(out_csv, index=False)

Found 111 candidate IDs.
Collected 10 rows. ObjectIDs: [437261, 459028, 459027, 437422, 438688, 436102, 438816, 436106, 435711, 436803]


,objectID,period,reign,artistDisplayName,artistRole,artistDisplayBio,artistNationality,artistBeginDate,artistEndDate,artistGender,...,additionalImages_count,objectDate,objectBeginDate,objectEndDate,metadataDate,medium,dimensions,culture,country,GalleryNumber
filled,10,10,10,10,10,10,10,10,10,10,...,10,10,10,10,10,10,10,10,10,10
missing,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Query European Decorative Arts department
One problem with using the query format, is that it produces non-deterministic behavior (issue #51, knonw issue 2). 

One could use the query ```q='""'```, which produces great, yet suboptimal results. For example using ```q='""'``` one gets ```37146``` results out of ```38681```. While using the query ```q=europe``` one woud get ```38568```.

In [46]:
EU_DEC = 12  # European Sculpture and Decorative Arts

# Search settings
search_query = '""'
search_filters = dict(
    departmentId=EU_DEC,
    hasImages=True,
    isPublicDomain=True,
)

SAMPLE_SIZE = 10

res = met_search(search_query, **search_filters)
ids = res.get("objectIDs") or []
print(f"Found {res.get('total', 0)} candidate IDs.")

rows, kept_ids = [], []
for oid in ids:
    o = met_object(oid)

    if not o.get("isPublicDomain"):
        continue
    if not (o.get("primaryImage") or o.get("primaryImageSmall")):
        continue

    rows.append(normalize_object(o))
    kept_ids.append(oid)
    if len(rows) == 10:
        break

df = pd.DataFrame(rows)
print(f"Collected {len(df)} rows. ObjectIDs: {kept_ids}")

filled = df.notna().sum()
missing = df.isna().sum()
summary = pd.DataFrame({"filled": filled, "missing": missing}).sort_values("missing", ascending=False)

display(summary.T)
out_csv = f"{CACHE_DIR}/met_decorative_arts_sample_10.csv"
df.to_csv(out_csv, index=False)

Found 37146 candidate IDs.
Collected 10 rows. ObjectIDs: [202614, 193875, 205709, 238518, 200377, 231856, 193843, 198002, 204588, 207712]


,tags,objectID,artistGender,reign,artistDisplayName,artistRole,artistDisplayBio,artistNationality,artistBeginDate,artistEndDate,...,additionalImages_count,objectDate,objectBeginDate,objectEndDate,metadataDate,medium,dimensions,culture,country,GalleryNumber
filled,9,10,10,10,10,10,10,10,10,10,...,10,10,10,10,10,10,10,10,10,10
missing,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Searching by Department ID
One other possibility is looking for all the objects in a department. However, this does not allow for checking if ```hasImage=true```, (which is anyway not perfect, issue #52, known issue 3), nor for ```isPublicDomain=true```. So every object has to be checked manually, which is not necessarily bad tho.

In [49]:
def met_objects_ids_from_dept(department_ids):
    if isinstance(department_ids, (list, tuple, set)):
        dep_str = "|".join(str(d) for d in department_ids)
    else:
        dep_str = str(department_ids)

    url = f"{BASE_URL}/objects"
    params = {"departmentIds": dep_str}

    cache_name = f"objects_{dep_str}"
    data = fetch_json(url, params=params, cache_name=cache_name)
    return data.get("objectIDs") or []

SAMPLE_SIZE = 100

dept_ids = met_objects_ids_from_dept(EU_DEC)
print(f"Dept {EU_DEC}: {len(dept_ids)} candidate IDs.")

rows, kept = [], []
for oid in dept_ids:
    o = met_object(oid)
    if not o:
        continue
    if not o.get("isPublicDomain"):
        continue
    if not (o.get("primaryImage") or o.get("primaryImageSmall")):
        continue
    rows.append(normalize_object(o))
    kept.append(oid)
    if len(rows) == SAMPLE_SIZE:
        break

df = pd.DataFrame(rows)
print(f"Collected {len(df)} rows from dept {EU_DEC}. ObjectIDs: {kept}")

filled = df.notna().sum()
missing = df.isna().sum()
summary = pd.DataFrame({"filled": filled, "missing": missing}).sort_values("missing", ascending=False)

display(summary.T)
out_csv = f"{CACHE_DIR}/met_decorative_arts_sample_10.csv"
df.to_csv(out_csv, index=False)

Dept 12: 43945 candidate IDs.
Collected 100 rows from dept 12. ObjectIDs: [13737, 13740, 16882, 79071, 79072, 79089, 79090, 79098, 79102, 79130, 79131, 79185, 108553, 108641, 155761, 155843, 159015, 159026, 159038, 159049, 159060, 159071, 159104, 172427, 173285, 173287, 173288, 173292, 173303, 173305, 173307, 173312, 173313, 173340, 173341, 186265, 186266, 186267, 186268, 186269, 186270, 186271, 186272, 186273, 186274, 186275, 186276, 186277, 186278, 186279, 186280, 186281, 186282, 186283, 186286, 186287, 186288, 186289, 186290, 186291, 186292, 186293, 186294, 186295, 186296, 186297, 186298, 186299, 186300, 186301, 186302, 186303, 186304, 186305, 186307, 186308, 186309, 186310, 186311, 186312, 186314, 186315, 186316, 186317, 186318, 186319, 186320, 186321, 186322, 186323, 186324, 186327, 186328, 186329, 186331, 186332, 186333, 186334, 186335, 186338]


,artistWikidata_URL,artistGender,artistDisplayName,artistRole,artistULAN_URL,tags,reign,artistDisplayBio,artistNationality,artistBeginDate,...,additionalImages_count,objectDate,objectBeginDate,objectEndDate,metadataDate,medium,dimensions,culture,country,GalleryNumber
filled,15,15,15,15,15,33,100,100,100,100,...,100,100,100,100,100,100,100,100,100,100
missing,85,85,85,85,85,67,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Sample 100 items from each department

In [ ]:
import re, random

SAMPLE_SIZE = 100
MAX_SCAN_PER_DEPT = 1000
OUTPUT_DIR = os.path.join(CACHE_DIR, "met_samples_by_department")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def slugify(name: str) -> str:
    s = re.sub(r"\s+", "_", name.strip())
    s = re.sub(r"[^\w\-\.]", "", s)
    return s.lower()[:120]

def met_departments():
    url = f"{BASE_URL}/departments"
    data = fetch_json(url, cache_name="departments")
    return data.get("departments", [])


departments = met_departments()
print(f"Found {len(departments)} departments.")

summary_rows = []

for d in departments:
    dep_id = d["departmentId"]
    dep_name = d["displayName"]

    ids = met_objects_ids_from_dept(dep_id)
    print(f"\n[dept {dep_id}] {dep_name} — {len(ids)} candidate IDs (pre-filter). Scanning...")

    random.shuffle(ids)

    rows, kept = [], []
    scan_limit = min(len(ids), MAX_SCAN_PER_DEPT)

    for oid in ids[:scan_limit]:
        o = met_object(oid)
        if not o:
            continue

        if not o.get("isPublicDomain"):
            continue
        if not (o.get("primaryImage") or o.get("primaryImageSmall")):
            continue

        rows.append(normalize_object(o))
        kept.append(oid)
        if len(rows) >= SAMPLE_SIZE:
            break

    if not rows:
        print(f"  -> No qualifying objects found (PD + image). Skipping CSV.")
        summary_rows.append({
            "departmentId": dep_id,
            "department": dep_name,
            "candidates": len(ids),
            "scanned": scan_limit,
            "kept": 0,
            "output_csv": None
        })
        continue

    df = pd.DataFrame(rows)

    out_name = f"dept_{dep_id}_{slugify(dep_name)}_sample_{len(df)}.csv"
    out_path = os.path.join(OUTPUT_DIR, out_name)
    df.to_csv(out_path, index=False)

    print(f"  -> Kept {len(df)} objects. Saved: {out_path}")

    filled = int(df.notna().sum().sum())
    missing = int(df.isna().sum().sum())
    summary_rows.append({
        "departmentId": dep_id,
        "department": dep_name,
        "candidates": len(ids),
        "scanned": scan_limit,
        "kept": len(df),
        "cells_filled": filled,
        "cells_missing": missing,
        "output_csv": out_path
    })

overview_df = pd.DataFrame(summary_rows)
overview_path = os.path.join(OUTPUT_DIR, "overview_by_department.csv")
overview_df.to_csv(overview_path, index=False)
print(f"\nOverview saved: {overview_path}")

try:
    display(overview_df)
except NameError:
    print(overview_df.to_string(index=False))